# Tong hop ket qua model
Notebook nay quet cac thu muc ket qua trong workspace va tong hop cac chi so:
- Acc
- MacroF1
- ClickF1 (F1 cua nhan 1)

Neu file ket qua khong co san metric, notebook se tu tinh tu file du lieu du doan neu co.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path.cwd()
records = []

def to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def add_record(model, acc, macro_f1, click_f1, source, note=""):
    records.append(
        {
            "Model": model,
            "Acc": to_float(acc),
            "MacroF1": to_float(macro_f1),
            "ClickF1": to_float(click_f1),
            "Source": str(source),
            "Note": note,
        }
    )

# 1) Bert_Fami: final_evaluation.json theo tung model
for fp in ROOT.glob("Bert_Fami/weights/*/final_evaluation.json"):
    try:
        data = json.loads(fp.read_text(encoding="utf-8"))
    except Exception:
        continue

    model_name = data.get("model_name", fp.parent.name)
    metrics = data.get("final_metrics") or data.get("best_checkpoint", {}).get("metrics", {})
    add_record(
        model=model_name,
        acc=metrics.get("accuracy"),
        macro_f1=metrics.get("macro_f1"),
        click_f1=metrics.get("clickbait_f1"),
        source=fp,
    )

# 2) Bert_Fami: evaluation_results.json (co the chua them model)
summary_fp = ROOT / "Bert_Fami/weights/evaluation_results.json"
if summary_fp.exists():
    try:
        data = json.loads(summary_fp.read_text(encoding="utf-8"))
        for item in data.get("models", []):
            model_name = item.get("model_name", "unknown_model")
            metrics = item.get("final_metrics") or item.get("best_metrics", {})
            add_record(
                model=model_name,
                acc=metrics.get("accuracy"),
                macro_f1=metrics.get("macro_f1"),
                click_f1=metrics.get("clickbait_f1"),
                source=summary_fp,
                note="from evaluation_results.json",
            )
    except Exception:
        pass

# 3) GPT-Shot: neu co predictions.csv thi tinh lai Acc/MacroF1/ClickF1
for pred_fp in ROOT.glob("GPT-Shot/*/predictions.csv"):
    try:
        dfp = pd.read_csv(pred_fp)
    except Exception:
        continue

    if {"true_label", "predicted_label"}.issubset(dfp.columns):
        y_true = pd.to_numeric(dfp["true_label"], errors="coerce")
        y_pred = pd.to_numeric(dfp["predicted_label"], errors="coerce")
        mask = y_true.notna() & y_pred.notna()
        y_true = y_true[mask].astype(int)
        y_pred = y_pred[mask].astype(int)

        if len(y_true) > 0:
            acc = accuracy_score(y_true, y_pred)
            macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
            click = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
            model_name = f"GPT-Shot/{pred_fp.parent.name}"
            add_record(model_name, acc, macro, click, pred_fp, note="computed from predictions.csv")

# 4) ORCD: S-teacher-bert.csv -> chon epoch co Accuracy cao nhat
for csv_fp in ROOT.glob("ORCD/*/weight/S-teacher-bert.csv"):
    try:
        dfo = pd.read_csv(csv_fp)
    except Exception:
        continue

    required_cols = {"Accuracy", "macro_f1", "true_f1"}
    if required_cols.issubset(dfo.columns) and len(dfo) > 0:
        best_idx = dfo["Accuracy"].astype(float).idxmax()
        best_row = dfo.loc[best_idx]
        model_name = f"ORCD/{csv_fp.parts[-3]}"
        add_record(
            model_name,
            best_row["Accuracy"],
            best_row["macro_f1"],
            best_row["true_f1"],
            csv_fp,
            note=f"best epoch={int(best_row['Epoch'])}" if "Epoch" in dfo.columns else "best row by Accuracy",
        )

# 5) SheepDog: chi co f1_macro trong summary, khong co click_f1 theo nhan 1
sheep_summary = ROOT / "SheepDog/results/clickbait_sheepdog_summary.json"
if sheep_summary.exists():
    try:
        data = json.loads(sheep_summary.read_text(encoding="utf-8"))
        m = data.get("original_test_avg", {})
        add_record(
            "SheepDog/summary",
            m.get("accuracy"),
            m.get("f1_macro"),
            np.nan,
            sheep_summary,
            note="ClickF1 khong co trong file ket qua",
        )
    except Exception:
        pass

# Tao dataframe tong hop
result_df = pd.DataFrame(records)

# Loai bo ban ghi trung model+source neu bi lap
if not result_df.empty:
    result_df = result_df.drop_duplicates(subset=["Model", "Source"], keep="first")
    result_df = result_df.sort_values(by=["Acc", "MacroF1", "ClickF1"], ascending=False, na_position="last")
    result_df[["Acc", "MacroF1", "ClickF1"]] = result_df[["Acc", "MacroF1", "ClickF1"]].round(4)

print(f"Tong so model/nguon ket qua: {len(result_df)}")
display(result_df)


Tong so model/nguon ket qua: 12


,Model,Acc,MacroF1,ClickF1,Source,Note
1,facebook/bart-large-mnli,0.9623,0.9582,0.9451,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,
4,facebook/bart-large-mnli,0.9623,0.9582,0.9451,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,from evaluation_results.json
0,FacebookAI/roberta-large,0.9540,0.9486,0.9321,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,
11,SheepDog/summary,0.9515,0.9460,NaN,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,ClickF1 khong co trong file ket qua
8,ORCD/GPT_3.5,0.9504,0.9435,0.9633,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,best epoch=11
10,ORCD/Wo_gpt,0.9504,0.9435,0.9633,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,best epoch=11
3,valurank/distilroberta-clickbait,0.9477,0.9419,0.9235,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,
5,valurank/distilroberta-clickbait,0.9477,0.9419,0.9235,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,from evaluation_results.json
9,ORCD/GPT_4o_mini,0.9461,0.9399,0.9592,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,best epoch=7
2,google-bert/bert-large-cased-whole-word-masking,0.9435,0.9378,0.9189,/mnt/c/Users/Admin/HUIT - Học Tập/Năm 3/Semest...,
